# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DjebrilSVN/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal check 1 — CTR vs Position (flag-linked: CTR-fix logic)**
FlyRank's CTR-fix flag fires when content ranks prominently on page 1 (`gsc_avg_position <= 10`) but captures below-expected clicks. In search engines, CTR drops steeply from top 3 to page 2+. The signal check tests whether page-1 items with below-benchmark CTR exist in meaningful volume. Verdict: **CONFIRMED** (details in code output below).

**Signal check 2 — Volume with zero clicks (flag-linked: quick-win logic)**
FlyRank's quick-win flag targets pages with visible search impressions (`gsc_impressions >= 100`) but exactly zero clicks. The signal check tests whether this pool of visible-but-ignored pages is sizable. Verdict: **CONFIRMED** (details in code output below).

**The rule (plain words):**
A content item earns an editorial review if it has measurable search presence (impressions >= 100 in March 2026) but is failing to capture organic clicks. Items ranking on page 1 with zero clicks represent the highest wasted opportunity and receive priority weighting (`baseline_score = impressions * (1 + page1_no_click)`). Items are assigned a single reason code explaining the primary opportunity.

**Reason codes:** `ctr_fix_and_quick_win` | `ctr_fix_opportunity` | `quick_win_opportunity` | `review_ctr`
**Action label:** `review_and_optimise_ctr`

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'], check=True)

import duckdb, os, json
import pandas as pd

# Load token securely from environment or .env
token = os.environ.get('HF_TOKEN')
if not token:
    for p in ['.env', '../.env', '../../.env']:
        if os.path.exists(p):
            for line in open(p):
                if line.strip().startswith('HF_TOKEN='):
                    token = line.strip().split('=', 1)[1]
            break
if not token and 'google.colab' in sys.modules:
    from google.colab import userdata
    try: token = userdata.get('HF_TOKEN')
    except Exception: pass

con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')"

# Aggregate to one row per content item (March 2026 slice)
df = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)  AS impressions,
        SUM(gsc_clicks)       AS clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM {REL}
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
""").df()

df['ctr'] = (df['clicks'] / df['impressions']) * 100

# ── Signal 1: CTR vs Position (flag-linked: CTR-fix logic) ────────────────
print('=== SIGNAL 1: CTR vs Position ===')
sig1 = df.copy()
sig1['pos_bucket'] = pd.cut(sig1['avg_position'],
                             bins=[0, 3, 10, 20, 50, float('inf')],
                             labels=['top3', 'page1', 'page2', 'page3_5', 'deep'])
bucket1 = sig1.groupby('pos_bucket', observed=True).agg(
    n=('ctr', 'count'),
    mean_ctr=('ctr', 'mean'),
    pct_zero_clicks=('clicks', lambda x: (x == 0).mean() * 100)
).reset_index()
print(bucket1.to_string(index=False))

# Benchmark: Active pages median CTR and underperforming page 1 volume
active_median_ctr = df[df['clicks'] > 0]['ctr'].median()
page1_low_ctr = ((df['avg_position'] <= 10) & (df['ctr'] < active_median_ctr) & (df['impressions'] >= 100)).sum()
page1_zero_clicks = ((df['avg_position'] <= 10) & (df['clicks'] == 0) & (df['impressions'] >= 100)).sum()
print(f'\nActive median CTR (for pages with >0 clicks): {active_median_ctr:.2f}%')
print(f'Page-1 pages with impressions >= 100 & CTR < active median: {page1_low_ctr:,}')
print(f'Page-1 pages with impressions >= 100 & exactly 0 clicks: {page1_zero_clicks:,}')
print('VERDICT: CONFIRMED — prominent rank gives visibility, but 14k+ page-1 items earn zero clicks.')

# ── Signal 2: Volume with zero clicks (flag-linked: quick-win logic) ──────
print('\n=== SIGNAL 2: Impressions >= 100, zero clicks ===')
sig2 = df.copy()
sig2['click_bucket'] = pd.cut(sig2['clicks'],
                               bins=[-1, 0, 5, 20, float('inf')],
                               labels=['zero', '1-5', '6-20', '20+'])
bucket2 = sig2[sig2['impressions'] >= 100].groupby('click_bucket', observed=True).agg(
    n=('impressions', 'count'),
    median_impressions=('impressions', 'median')
).reset_index()
print(bucket2.to_string(index=False))
quick_win_n = ((df['impressions'] >= 100) & (df['clicks'] == 0)).sum()
print(f'\nn pages with impressions >= 100 and zero clicks: {quick_win_n:,}')
print('VERDICT: CONFIRMED — massive pool of 37k+ visible pages receiving zero clicks.')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== SIGNAL 1: CTR vs Position ===
pos_bucket     n  mean_ctr  pct_zero_clicks
      top3 16144  1.058938        51.629088
     page1 81987  0.492611        54.966031
     page2 32204  0.321109        59.328034
   page3_5 33288  0.228715        68.751502
      deep 11681  0.090312        95.300060

Active median CTR (for pages with >0 clicks): 0.32%
Page-1 pages with impressions >= 100 & CTR < active median: 35,588
Page-1 pages with impressions >= 100 & exactly 0 clicks: 14,561
VERDICT: CONFIRMED — prominent rank gives visibility, but 14k+ page-1 items earn zero clicks.

=== SIGNAL 2: Impressions >= 100, zero clicks ===
click_bucket     n  median_impressions
        zero 37779               283.0
         1-5 37982               842.0
        6-20 15999              3107.0
         20+  9681              8104.0

n pages with impressions >= 100 and zero clicks: 37,779
VERDICT: CONFIRMED — massive pool of 37k+ visible pages receiving zero clicks.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Only pages with measurable visibility (impressions >= 100)
queue = df[df['impressions'] >= 100].copy()

# Binary signal indicators (strictly past observations)
queue['page1_no_click'] = ((queue['avg_position'] <= 10) & (queue['clicks'] == 0)).astype(int)
queue['high_imp_no_click'] = ((queue['impressions'] >= 100) & (queue['clicks'] == 0)).astype(int)

# Score: impressions at waste, doubled if wasted on page 1
queue['baseline_score'] = queue['impressions'] * (1 + queue['page1_no_click'])

# Reason codes (one code per row, most specific condition wins)
def assign_reason(row):
    if row['page1_no_click'] and row['high_imp_no_click']:
        return 'ctr_fix_and_quick_win'
    elif row['page1_no_click']:
        return 'ctr_fix_opportunity'
    elif row['high_imp_no_click']:
        return 'quick_win_opportunity'
    else:
        return 'review_ctr'

queue['reason_code'] = queue.apply(assign_reason, axis=1)
queue['action_label'] = 'review_and_optimise_ctr'

# Rank by baseline score descending
queue = queue.sort_values('baseline_score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

# Robust output path resolution regardless of current working directory
_cwd = os.getcwd()
if os.path.exists(os.path.join(_cwd, 'work', 'outputs')) or os.path.exists(os.path.join(_cwd, 'work', 'notebooks')):
    _out_dir = os.path.join(_cwd, 'work', 'outputs')
elif os.path.basename(_cwd) == 'notebooks':
    _out_dir = os.path.abspath(os.path.join(_cwd, '..', 'outputs'))
elif os.path.basename(_cwd) == 'work':
    _out_dir = os.path.abspath(os.path.join(_cwd, 'outputs'))
else:
    _out_dir = os.path.abspath('work/outputs')
os.makedirs(_out_dir, exist_ok=True)

out_cols = ['rank', 'client_hash_id', 'content_hash_id',
            'impressions', 'clicks', 'avg_position', 'ctr',
            'baseline_score', 'reason_code', 'action_label']

# Write queue CSV (gitignored by design)
queue[out_cols].to_csv(os.path.join(_out_dir, 'baseline_action_score.csv'), index=False)

# Write metrics JSON (committed run receipt)
metrics = {
    'total_scored': int(len(queue)),
    'reason_code_counts': queue['reason_code'].value_counts().to_dict(),
    'median_score': float(queue['baseline_score'].median()),
    'top1_score': float(queue['baseline_score'].iloc[0])
}
with open(os.path.join(_out_dir, 'baseline_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Queue: {len(queue):,} pages scored and ranked')
print(f'CSV written to {os.path.join(_out_dir, "baseline_action_score.csv")}')
print(f'Metrics written to {os.path.join(_out_dir, "baseline_metrics.json")}')
print('\nReason code breakdown:')
print(queue['reason_code'].value_counts().to_string())


Queue: 101,441 pages scored and ranked
CSV written to C:\Users\djebr\Documents\GitHub\flyrank-ml-internship\work\outputs\baseline_action_score.csv
Metrics written to C:\Users\djebr\Documents\GitHub\flyrank-ml-internship\work\outputs\baseline_metrics.json

Reason code breakdown:
reason_code
review_ctr               63662
quick_win_opportunity    23218
ctr_fix_and_quick_win    14561


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 Review with Skepticism (Action | Why it's there | What would make it wrong)

1. **Rank 1 (`content_eadb33b5df496f4a`):** `review_and_optimise_ctr` | **Why it scored:** Massive volume (617k impressions at pos 2.4). | **What would make it wrong:** Already earns 5,668 clicks (champion page); raw impressions dominated the simple score, making this a false positive for an underperformance queue.
2. **Rank 2 (`content_ec2e0346994fb5a5`):** `review_and_optimise_ctr` | **Why it scored:** 245k impressions at pos 2.9 with 1,480 clicks. | **What would make it wrong:** Navigational/brand query with healthy organic CTR; rewriting snippet copy risks destabilizing top-3 position.
3. **Rank 3 (`content_e8a52cf3d5988c07`):** `review_and_optimise_ctr` | **Why it scored:** 245k impressions ranking on Page 2 (pos 15.0). | **What would make it wrong:** Ranking on Page 2 naturally depresses CTR (0.27%); title/meta fixes cannot generate clicks without rank first moving to Page 1.
4. **Rank 4 (`content_0e03de7680314cd5`):** `review_and_optimise_ctr` | **Why it scored:** Top-3 rank (pos 2.7) with 221k impressions but only 0.33% CTR. | **What would make it wrong:** Zero-click SERP features (featured snippets, AI Overviews) answering query intent directly without clicks.
5. **Rank 5 (`content_44f34c0a90047651`):** `review_and_optimise_ctr` | **Why it scored:** Severe underperformance (212k impressions on page 1 with only 24 clicks, 0.01% CTR). | **What would make it wrong:** Accidental ranking for a broad, non-commercial search intent that doesn't align with page content.
6. **Rank 6 (`content_7172a7fad43f0998`):** `review_and_optimise_ctr` | **Why it scored:** Pos 3.4 with 206k impressions and 0.42% CTR. | **What would make it wrong:** Competitive commercial SERP where Google Ads push organic snippet below fold.
7. **Rank 7 (`content_e7b5dd4dff461ad2`):** `review_and_optimise_ctr` | **Why it scored:** 205k impressions at pos 4.5. | **What would make it wrong:** Page already earns 2,446 clicks (1.19% CTR, above pos-4 average); flagged purely on volume alone.
8. **Rank 8 (`content_8d7d99f109e19aa2`):** `review_and_optimise_ctr` | **Why it scored:** Top-3 rank (pos 2.6) with 203k impressions and only 289 clicks (0.14% CTR). | **What would make it wrong:** Snippet truncation or missing rich schema in search results causing lower click-through.
9. **Rank 9 (`content_f107e54b10b43725`):** `review_and_optimise_ctr` | **Why it scored:** Pos 3.2 with 196k impressions and 996 clicks (0.51% CTR). | **What would make it wrong:** Informational definition query where users read the quick answer on the SERP rather than clicking.
10. **Rank 10 (`content_36e53e9c707674fc`):** `review_and_optimise_ctr` | **Why it scored:** 195k impressions ranking on Page 4 (pos 32.8). | **What would make it wrong:** Buried on Page 4; poor CTR (0.12%) is an artifact of position depth rather than poor editorial metadata.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top10 = queue[out_cols].head(10)
print('=== TOP 10 RANKED PAGES ===')
print(top10.to_string(index=False))


=== TOP 10 RANKED PAGES ===
 rank          client_hash_id          content_hash_id  impressions  clicks  avg_position      ctr  baseline_score reason_code            action_label
    1 client_e547b89c05043229 content_eadb33b5df496f4a     617124.0  5668.0      2.383011 0.918454        617124.0  review_ctr review_and_optimise_ctr
    2 client_e547b89c05043229 content_ec2e0346994fb5a5     245276.0  1480.0      2.854514 0.603402        245276.0  review_ctr review_and_optimise_ctr
    3 client_23a62021009f63c4 content_e8a52cf3d5988c07     244931.0   669.0     15.008339 0.273138        244931.0  review_ctr review_and_optimise_ctr
    4 client_e547b89c05043229 content_0e03de7680314cd5     221310.0   720.0      2.675217 0.325336        221310.0  review_ctr review_and_optimise_ctr
    5 client_23a62021009f63c4 content_44f34c0a90047651     212404.0    24.0      7.346909 0.011299        212404.0  review_ctr review_and_optimise_ctr
    6 client_62f4a7e64f5e0096 content_7172a7fad43f0998     205867.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks identified in the top queue:**
1. **Volume false positives (e.g. Rank 1 & Rank 7):** Pages that already earn thousands of clicks (5,668 and 2,446 clicks) sit at the top of the queue simply because their raw impressions exceed 200k. The baseline rule rewards sheer volume rather than inefficiency ratio.
2. **Deep position artifacts (e.g. Rank 3 & Rank 10):** Pages ranking on page 2 or page 4 (positions 15.0 and 32.8) have low CTR simply because of position bias, not bad copy. Assigning them `review_and_optimise_ctr` without first improving page rank is an ineffective action.
3. **Position anomaly rows (`avg_position < 1.0`):** In GSC data, avg_position below 1.0 reflects edge case reporting anomalies. The baseline scores them if impressions are high, which a refined ML model must filter out.

**Leakage check:**
The baseline rule relies strictly on historical aggregate metrics from March 2026 (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`). No target labels (`is_declining_label`), no future-window measurements (April 2026+), and no product-derived flags are used.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Audit weak picks: pages with high clicks flagged for 'review_ctr' (volume false positives)
volume_false_positives = queue[(queue['clicks'] >= 1000) & (queue['rank'] <= 20)]
print(f'Weak picks in Top 20 (High-click pages scored purely on volume): {len(volume_false_positives)}')
if len(volume_false_positives):
    print(volume_false_positives[['rank', 'impressions', 'clicks', 'avg_position', 'ctr', 'reason_code']].to_string(index=False))

# 2. Audit position anomalies (avg_position < 1.0 despite high impressions)
pos_anomalies = queue[(queue['avg_position'] < 1.0) & (queue['impressions'] >= 500)].head(5)
print(f'\nPosition anomalies (avg_position < 1.0 with imp >= 500): {len(pos_anomalies)}')
if len(pos_anomalies):
    print(pos_anomalies[['rank', 'impressions', 'clicks', 'avg_position', 'reason_code']].to_string(index=False))

# 3. Strict leakage assertion: verify only permitted historical columns were touched
used_cols = set(queue.columns)
forbidden_patterns = ['future', 'next', 'is_declining', 'trend_direction', 'trend_pct', 'target_label']
leaked = [col for col in used_cols if any(pat in col.lower() for pat in forbidden_patterns)]
assert len(leaked) == 0, f'Leakage detected! Found forbidden columns: {leaked}'

print('\nLeakage check: PASSED (0 leaked columns). All inputs are past March 2026 aggregates.')


Weak picks in Top 20 (High-click pages scored purely on volume): 4
 rank  impressions  clicks  avg_position      ctr reason_code
    1     617124.0  5668.0      2.383011 0.918454  review_ctr
    2     245276.0  1480.0      2.854514 0.603402  review_ctr
    7     205045.0  2446.0      4.544203 1.192909  review_ctr
   15     154358.0  2506.0      3.019798 1.623499  review_ctr

Position anomalies (avg_position < 1.0 with imp >= 500): 5
 rank  impressions  clicks  avg_position reason_code
  714      35426.0    46.0      0.596619  review_ctr
  746      34700.0    70.0      0.761700  review_ctr
 1068      28753.0    82.0      0.544579  review_ctr
 1359      25429.0    19.0      0.952650  review_ctr
 3006      16005.0    29.0      0.750536  review_ctr

Leakage check: PASSED (0 leaked columns). All inputs are past March 2026 aggregates.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.